# ML-02 — Research Question and Provisional Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Yashcodes07/flyrank-ml-internship/blob/main/work/notebooks/w01_research_question.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [4]:
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/Yashcodes07/flyrank-ml-internship"
REPO_DIR = "flyrank-ml-internship"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
else:
    while not os.path.isdir("data/raw") and os.getcwd() != "/":
        os.chdir("..")

print("Working dir:", os.getcwd())

Working dir: /content/flyrank-ml-internship


## 1. My lane (or freestyle) and why

*Name your lane — or say 'freestyle' and describe your own question. One short paragraph: why this one?*

I'm picking Lane 2 — Refresh / Content Opportunity Scoring. The starter pipeline already shows a learned model meaningfully outperforming a hand-written rule at this task (baseline Precision@50 = 0.240 vs random forest = 0.740), which tells me there's real signal here worth 7 more weeks of work, not just noise.


In [1]:
print("Lane selected: Refresh / Content Opportunity Scoring")
print("Baseline rules Precision@50: 0.240")
print("Random forest Precision@50: 0.740")


Lane selected: Refresh / Content Opportunity Scoring
Baseline rules Precision@50: 0.240
Random forest Precision@50: 0.740


## 2. The question: decision, action, cost of a wrong call

*What decision does your work improve? Who acts on it? What does a wrong recommendation cost?*

**Decision:** Which content pages should be reviewed first for a refresh,
out of a large inventory, given limited reviewer capacity.

**Unit of analysis:** One content page (`content_id`), per client.

**Action:** A content reviewer schedules a refresh, rewrite, or expansion
for the page.

**Cost of a wrong call:**
- False positive → a reviewer spends limited hours on a page that wasn't
  actually declining, wasting capacity that could've gone to a real problem.
- False negative → a genuinely declining, high-demand page goes unreviewed
  and keeps losing visibility and traffic.

**Why data/ML can help:** A hand-written baseline rule only got about 24%
of its top 50 picks right (Precision@50 = 0.240). A learned model got about
74% right (0.740) on the same task — roughly 3x more true positives in the
same review budget. That gap is too large to be a fluke, and it means a
simple rule leaves real value on the table that a model can recover.

In [5]:
import pandas as pd
df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
print(df["trend_direction"].value_counts())
print(f"\nTotal pages in starter dataset: {len(df)}")


trend_direction
down      16262
stable     5962
up         4388
new        2236
flat       1152
Name: count, dtype: int64

Total pages in starter dataset: 30000


## 3. Quick look at the data (2-3 real numbers)

*Load the starter CSV below and show 2-3 real numbers that make your lane look worth the next 7 weeks.*

Three real numbers from the starter dataset that support this lane:
1. Model vs. baseline lift (from the starter pipeline's own results)
2. CTR drops sharply by position tier — supports the `low_ctr_visible_page`
   reason code this lane's baseline uses
3. Declining pages skew younger, not older — a non-obvious pattern worth
   investigating further in later weeks

In [6]:
import pandas as pd, numpy as np
df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

# Number 1: baseline vs model lift (verified from outputs/model_report.md)
print("Baseline rules Precision@50: 0.240")
print("Random forest Precision@50: 0.740")
print("Lift: ~3x more true positives in the top 50 vs the hand-written rule\n")

# Number 2: CTR by position tier
visible = df[df["impressions_90d"] >= 100]
ctr_by_pos = visible.groupby("position_tier")["ctr"].mean().sort_values(ascending=False)
print("Mean CTR by position tier (impressions >= 100):")
print(ctr_by_pos.round(4).to_string())

# Number 3: content age by trend direction
age_trend = df.groupby("trend_direction")["content_age_days"].median()
print("\nMedian content age by trend direction:")
print(age_trend.round(0).to_string())


Baseline rules Precision@50: 0.240
Random forest Precision@50: 0.740
Lift: ~3x more true positives in the top 50 vs the hand-written rule

Mean CTR by position tier (impressions >= 100):
position_tier
page_1      0.3548
top_3       0.3341
striking    0.2558
page_3_5    0.1424
deep        0.0554

Median content age by trend direction:
trend_direction
down      216.0
flat      231.0
new       279.0
stable    300.0
up        292.0


## 4. Careful words: what I can and can't claim

*Write what your work will be able to say (observed, directional, decision-support) — and what it never will (causal proof, 'predicting Google').*

**What I can claim:**
- Certain observable signals (position, CTR, content age, visibility trend)
  are *associated* with pages later flagged as declining in this dataset.
- A ranked review queue built from these signals can help a human reviewer
  prioritize limited time more effectively than a simple rule.
- This is decision-support: a ranked list for a human to review, not an
  automated decision.

**What I can't claim:**
- That refreshing a page will *cause* it to recover — that requires a
  controlled experiment, not observational data.
- That `trend_direction == "down"` is a true predictive label — it's a
  beginner proxy based on the *current* window, not a genuine future
  outcome. A stronger capstone version would define decline using a
  future window (e.g. prior 90 days predicting the next 30 days).
- That any reported precision number is trustworthy without proper
  client-holdout validation. My own Notebook 02 experiment showed a 0.200
  gap between train and test precision on a naive random split — in-sample
  numbers alone overstate real performance.
- That I've discovered anything about Google's actual ranking algorithm —
  only associations within FlyRank's own anonymized data.

In [7]:
print("See above for this section")

See above for this section


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.